# DZ10 — Supervised ML: Економічне моделювання
## **Завдання:** з’ясувати, чи підвищення ціни на апельсини спричиняє зростання попиту на яблука?

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# 1) Завантаження даних
df = pd.read_csv('orange-apple.csv')
# Перегляд даних (перші 5 записів)
print(df.head())

   orange_price  apple_quantity
0        210.43             483
1         94.80             281
2        277.37             618
3         97.48             258
4        114.38             303


Розпочинаємо дослідження з імпорту потрібних бібліотек:  
- **pandas** — для роботи з таблицями даних;  
- **LinearRegression** — реалізація моделі лінійної регресії;  
- **train_test_split** — поділ даних на навчальну та тестову вибірки;  
- **mean_squared_error**, **r2_score** — для оцінки точності моделі;  
- **pearsonr** — для обчислення коефіцієнта кореляції між змінними.

Завантажуємо дані з файлу **orange-apple.csv**, який містить два показники:
- **orange_price** — ціна на апельсини (незалежна змінна),
- **apple_quantity** — кількість продажів яблук (цільова змінна).

In [2]:
# 2) Формування незалежної та цільової ознак
X = df[['orange_price']]       # незалежна змінна
y = df[['apple_quantity']]     # цільова змінна

print("X (ціна на апельсини):")
print(X.head())

print("\ny (попит на яблука):")
print(y.head())

X (ціна на апельсини):
   orange_price
0        210.43
1         94.80
2        277.37
3         97.48
4        114.38

y (попит на яблука):
   apple_quantity
0             483
1             281
2             618
3             258
4             303


Визначаємо історичні дані - **ознаки (features)** і **цільову змінну (target)**:

- `X` — це те, що впливає (ціна апельсинів, незалежна змінна);  
- `y` — це те, що ми хочемо передбачити (кількість проданих яблук, цільова змінна).

Таким чином, ми перевіряємо, чи впливає підвищення ціни на апельсини на збільшення попиту на яблука.  
Якщо зв’язок є — це означає, що яблука є **замінниками** апельсинів.

In [3]:
# 3) Розподіл даних на тренувальну і тестову вибірки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Розміри вибірок:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Розміри вибірок:
X_train: (80, 1)
X_test: (20, 1)


Ділимо дані на:
- **Навчальну вибірку (train)** — для навчання моделі (80% даних),
- **Тестову вибірку (test)** — для перевірки її точності (20% даних).

Такий поділ дозволяє уникнути перенавчання нашої моделі.  
Параметр `random_state=42` забезпечує стабільність і повторюваність результатів.

In [4]:
# 4) Побудова та навчання моделі
model = LinearRegression()
model.fit(X_train, y_train)

# 5) Отримуємо коефіцієнти моделі
slope = model.coef_[0][0]
intercept = model.intercept_[0]

print("SLOPE:", slope)
print("INTERCEPT:", intercept)

SLOPE: 1.8402013948218765
INTERCEPT: 92.92078454769347


Модель **лінійної регресії** будує рівняння вигляду:  
**y = a0 + a1 * x**  
де:
- `a1` (**SLOPE**) — показує, наскільки змінюється обсяг продажів яблук при зміні ціни апельсинів на 1 одиницю;
- `a0` (**INTERCEPT**) — початкове значення продажів яблук, якщо ціна апельсинів = 0.

Якщо `SLOPE > 0`, то між цими товарами існує **позитивна залежність**:  
чим дорожче апельсини — тим більше купують яблук.

In [5]:
# 6) Прогнозування та оцінка моделі
y_pred = model.predict(X_test)

# 7) Обчислення метрик
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** (1/2)
corr, p_value = pearsonr(df['orange_price'], df['apple_quantity'])

print("R-SQUARED:", r2)
print("RMSE:", rmse)
print("CORRELATION:", corr)
print("p-value:", p_value)

R-SQUARED: 0.9907743736163998
RMSE: 10.099407950553323
CORRELATION: 0.9943675329067552
p-value: 2.411637116525437e-97


**R-SQUARED (R²)** — коефіцієнт детермінації:  
показує, яку частину змін у продажах яблук пояснює ціна апельсинів.  

**RMSE (Root Mean Squared Error)** — середня похибка моделі:  
показує, на скільки одиниць у середньому модель помиляється у прогнозі.  

**CORRELATION** — коефіцієнт кореляції Пірсона:  
якщо `значення > 0.7` — сильний позитивний зв’язок;  
якщо `p-value < 0.05` — цей зв’язок статистично значущий, тобто не випадковий.

In [6]:
# 8) Прогноз попиту при новій ціні на апельсини
next_ = pd.DataFrame({'orange_price': [3.2]})
prediction = model.predict(next_)

print(f"Прогнозований попит на яблука, при ціні апельсинів `3.2` = {prediction[0][0]}")

Прогнозований попит на яблука, при ціні апельсинів `3.2` = 98.80942901112348


Тут ми застосовуємо нашу модель для передбачення майбутнього:  
Якщо **ціна на апельсини зросте до 3.2**, ми можемо оцінити очікуваний попит на яблука.  
Це практична частина економічного прогнозування — така модель може допомогти торговим мережам  
планувати закупівлі і баланс між схожими товарами.

In [7]:
# 9) Відповідь
print(f"""SLOPE       = {slope:.6f}
INTERCEPT   = {intercept:.6f}
R-SQUARED   = {r2:.6f}
RMSE        = {rmse:.6f}
CORRELATION = {corr:.6f}
(p_value = {p_value:.99f})
""".rstrip())

SLOPE       = 1.840201
INTERCEPT   = 92.920785
R-SQUARED   = 0.990774
RMSE        = 10.099408
CORRELATION = 0.994368
(p_value = 0.000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000241)


## Дослідження (інтерпретація результатів)

Ми провели економічне дослідження, щоб перевірити, чи впливає підвищення ціни на апельсини на попит на яблука.  
Після побудови та навчання моделі **лінійної регресії**, ми отримали такі результати:  

### SLOPE (коефіцієнт нахилу)

`SLOPE = 1.840201` — Це означає, що при підвищенні ціни на апельсини на 1 одиницю, середній попит на яблука зростає на **1.84** одиниці.  
Коефіцієнт нахилу показує, наскільки швидко змінюється **попит (y)** при зміні **ціни на апельсини (x)**.  
Оскільки він додатній, то зв’язок між змінними прямий, тобто, чим дорожчі апельсини, тим більше люди купують яблука.  
Це типова економічна ситуація, коли товари є замінниками (substitute goods).  
Таким чином, яблука дійсно виступають альтернативним товаром для споживачів, коли ціна на апельсини підвищується.  

---

### INTERCEPT (вільний член)

`INTERCEPT = 92.920785` — Це **теоретичний базовий рівень продажів яблук**, коли ціна на апельсини дорівнює нулю.
Тобто, навіть якщо апельсини будуть безкоштовні (гіпотетично), **очікуваний попит на яблука становитиме близько 93 одиниці**.
Цей параметр визначає **точку**, **де наша регресійна пряма перетинає вісь Y**, і задає початкове положення моделі на графіку.
Економічно це можна тлумачити як **мінімальний або базовий попит на яблука**, який існує незалежно від зовнішніх  
цінових факторів — наприклад, через смакові переваги, традиційний попит чи інші нефінансові мотиви.

---

### R-SQUARED (коефіцієнт детермінації)

`R-SQUARED = 0.990774` — Це означає, що **99.08% усіх змін у попиті на яблука (коли продажі зростають або падають), пояснюються змінами ціни на апельсини**. Решта **приблизно 1%** припадає на інші фактори (сезонність, реклама, знижки, свята, погодні умови тощо).  
**R²** — це показник якості моделі, який відповідає на запитання:  
>"Наскільки добре лінійна модель пояснює реальні дані?"  

Значення **0.99** — це **дуже високий рівень пояснювальної здатності**.  
Модель майже ідеально відображає залежність між змінними, тобто ціна апельсинів дійсно майже повністю визначає попит на яблука.  
У реальних економічних даних такий високий показник зустрічається рідко, тому результат можна вважати **виключно якісним**.  

---

### RMSE (Root Mean Squared Error), середня похибка для нашої моделі

`RMSE = 10.099408` — Це **середнє відхилення між реальними значеннями попиту на яблука та тими**, **які прогнозує наша модель**.  
Інакше кажучи, **у середньому, модель помиляється лише на ≈ 10 одиниць продажів**.  
**RMSE** — це *“міра точності”* прогнозу:  
Чим менше **RMSE**, тим точніше модель. Якщо **RMSE** невелике порівняно з масштабом даних, модель вважається **адекватною і надійною**.  
Отже, при **R² = 0.99** і **RMSE ≈ 10**, можна стверджувати, що **модель передбачає попит дуже точно**, а похибка є незначною з практичної точки зору.  

---

### CORRELATION (“коефіцієнт Пірсона”)

`CORRELATION = 0.994368` — Це показник **сили та напряму лінійного зв’язку** між ціною апельсинів і попитом на яблука.  
Він завжди лежить у межах від **-1** до **+1**.  
Якщо **+1** → ідеальний прямий зв’язок (чим більше `X`, тим більше `y`);  
Якщо **0** → зв’язок відсутній;  
Якщо **-1** → ідеальний зворотний зв’язок.  
Наш коефіцієнт **0.994** майже дорівнює **1**, що вказує на **майже ідеальний прямий зв’язок**:  
збільшення ціни на апельсини **сильно і прямо** впливає на зростання продажів яблук.  
Додатково, `p-value ≈ 2.41e-100` свідчить, що **ймовірність випадкового отримання такого зв’язку практично дорівнює нулю**.  
Отже, зв’язок **статистично значущий** — він є реальним, а не випадковим ефектом у даних.  

---

### Прогноз попиту при новій ціні на апельсини

***Модель також дозволяє зробити практичне передбачення.***  
Припустимо, якщо ціна на апельсини підвищиться, то за допомогою нашого рівняння регресії:  
**y = 92.920785 + 1.840201 × (ціна_апельсинів)**, ми можемо легко передбачити очікуваний попит на яблука.  
Наприклад, якщо **ціна на апельсини становить 3.2**, то очікуваний попит складе приблизно **y = 92.92 + 1.84 × 3.2 ≈ 98.81 одиниць**.  
Це підтверджує, що при підвищенні ціни на апельсини, споживачі **переходять на яблука**, **як на альтернативний товар**.  

## Підсумковий висновок  
### Отримані результати показують, що:  

1) Між ціною на апельсини та попитом на яблука існує **сильна позитивна залежність**;  
2) **Підвищення ціни на апельсини** призводить до **зростання попиту на яблука**;  
3) **Модель має надзвичайно високу точність (R² = 0.99)** і **мінімальну похибку (RMSE ≈ 10)**;  
4) **Зв’язок між змінними є статистично достовірним (p < 0.001)**.  

Таким чином, економічна гіпотеза повністю підтверджена — **яблука є замінниками апельсинів (substitute goods)**, 
а побудована модель може бути використана для **прогнозування попиту та аналізу поведінки споживачів у роздрібній торгівлі**.